In [1]:
import os
import warnings

import agama
import astropy
import gc_utils
import gizmo_analysis as gizmo
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import utilities as ut
from matplotlib.animation import PillowWriter
from matplotlib.colors import LogNorm, Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy import stats
from scipy.interpolate import interp1d
from scipy.optimize import curve_fit, minimize
from scipy.signal import find_peaks
from scipy.stats import gaussian_kde
from sklearn.svm import SVC

agama.setUnits(mass=1, length=1, velocity=1)
units = agama.getUnits()

KeyboardInterrupt: 

In [ ]:
sim = "m12i"
sim_dir = "/Users/z5114326/Documents/simulations/"

ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
ghost_data = h5py.File(ghost_file, "r")

proc_file = sim_dir + sim + "/" + sim + "_processed.hdf5"
proc_data = h5py.File(proc_file, "r")

in_situ_msk = True  # True for include in_situ
ex_situ_msk = False  # True for include ex_situ

snap_min = 46

In [ ]:
public_snapshot_file = sim_dir + "snapshot_times_public.txt"
pub_data = pd.read_table(public_snapshot_file, comment="#", header=None, sep=r"\s+")
pub_data.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]

snap_lst = np.array([snap for snap in pub_data["index"] if snap >= snap_min])
# snap_lst = snap_lst[-3:]
snap_lst

array([ 46,  52,  59,  67,  77,  88, 102, 120, 142, 172, 214, 277, 294,
       312, 332, 356, 382, 412, 446, 486, 534, 590, 591, 592, 593, 594,
       595, 596, 597, 598, 599, 600])

In [ ]:
def sph_to_xyz(posvel_sph):
    """
    Convert 6D spherical (r, theta, phi, vr, vtheta, vphi)
    to 6D Cartesian (x, y, z, vx, vy, vz).

    Parameters
    ----------
    posvel_sph : array-like, shape (6,)
        [r, theta, phi, vr, vtheta, vphi]
        theta is the polar angle (colatitude).

    Returns
    -------
    posvel_cart : ndarray, shape (6,)
        [x, y, z, vx, vy, vz]
    """
    # r, theta, phi, vr, vtheta, vphi = posvel_sph
    r, theta, phi, vr, vtheta, vphi = np.moveaxis(posvel_sph, -1, 0)

    sin_t = np.sin(theta)
    cos_t = np.cos(theta)
    sin_p = np.sin(phi)
    cos_p = np.cos(phi)

    # Position
    x = r * sin_t * cos_p
    y = r * sin_t * sin_p
    z = r * cos_t

    # Velocity
    vx = vr * sin_t * cos_p + vtheta * cos_t * cos_p - vphi * sin_p
    vy = vr * sin_t * sin_p + vtheta * cos_t * sin_p + vphi * cos_p
    vz = vr * cos_t - vtheta * sin_t

    # return np.array([x, y, z, vx, vy, vz])
    return np.stack((x, y, z, vx, vy, vz), axis=-1)


def get_period(posvel, pot_nbody):
    result = agama.orbit(potential=pot_nbody, ic=posvel, time=10 * pot_nbody.Tcirc(posvel), trajsize=1000)

    t_orb = []
    for result_i in result:
        t_i_agama = result_i[0]
        t_i_myr = t_i_agama * units["time"]  # Myr
        orb_i = result_i[1]

        r_i = np.linalg.norm(orb_i[:, 0:3], axis=1)
        peaks, _ = find_peaks(r_i)

        if len(peaks) < 2:
            t_orb_i = np.nan
        else:
            t_orb_i = np.mean(np.diff(t_i_myr[peaks]))

        t_orb.append(t_orb_i)
    return np.array(t_orb)

In [ ]:
metadata = dict(title="t_mass_trends_with_time", artist="Finn")
writer = PillowWriter(fps=2, metadata=metadata)

fig = plt.figure(figsize=(8, 6))

gif_loc = sim_dir + sim + "/gifs/"
if not os.path.exists(gif_loc):
    os.makedirs(gif_loc)


gif_file = gif_loc + "t_mass_trends_with_time.gif"
with writer.saving(fig, gif_file, 100):
    for snap in snap_lst:
        snap_id = gc_utils.snapshot_name(snap)

        t_snap = pub_data["time_Gyr"][pub_data["index"] == snap].values[0]
        t_snap = np.round(t_snap, 4)

        pot_file = sim_dir + sim + "/potentials/snap_%d/combined_snap_%d.ini" % (snap, snap)
        pot_nbody = agama.Potential(pot_file)

        torb = np.array([])
        mass = np.array([])

        for it_id in ghost_data.keys():
            print(snap_id, "-", it_id)
            snp_dat = ghost_data[it_id]["snapshots"][snap_id]

            amsk = snp_dat["grpid"][()] == 0
            if (in_situ_msk) & (not ex_situ_msk):
                msk = amsk
            elif (not in_situ_msk) & (ex_situ_msk):
                msk = ~amsk
            else:
                msk = np.ones(len(amsk), dtype=bool)

            pos_sph = snp_dat["host.pos.sph"][msk]
            vel_sph = snp_dat["host.vel.sph"][msk]
            log_mass = snp_dat["logm"][msk]
            mass_i = 10**log_mass

            posvel_sph = np.column_stack((pos_sph, vel_sph))
            posvel_xyz = sph_to_xyz(posvel_sph)

            torb_i = get_period(posvel_xyz, pot_nbody)

            torb = np.concatenate((torb, torb_i))
            mass = np.concatenate((mass, mass_i))

        ax = plt.subplot2grid(shape=(1, 1), loc=(0, 0))
        sc = ax.scatter(mass, torb, c="r", s=10)

        ax.set_xlabel(r"GC Mass [M$_{\odot}$]")
        ax.set_ylabel(r"Orbital Period (Myr)")

        ax.set_xscale("log")
        ax.set_yscale("log")

        ax.set_xlim(10**-1, 10**7)
        ax.set_ylim(10**-1, 10**4.2)

        time_stamp = "Time: " + str(t_snap) + " Gyr"
        ax.text(
            0.70,
            0.95,
            time_stamp,
            color="black",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=12,
        )

        writer.grab_frame()
        fig.clear()

8 orbits complete (2000 orbits/s)
8 orbits complete (2000 orbits/s)
5 orbits complete (1667 orbits/s)
5 orbits complete (2500 orbits/s)
13 orbits complete (2600 orbits/s)
6 orbits complete (857.1 orbits/s)
9 orbits complete (900 orbits/s)
4 orbits complete (666.7 orbits/s)
7 orbits complete (700 orbits/s)
5 orbits complete (1667 orbits/s)
5 orbits complete (1667 orbits/s)
9 orbits complete (1125 orbits/s)
4 orbits complete (2000 orbits/s)
6 orbits complete (3000 orbits/s)
11 orbits complete (3667 orbits/s)
10 orbits complete (5000 orbits/s)
4 orbits complete (4000 orbits/s)
13 orbits complete (4333 orbits/s)
13 orbits complete (4333 orbits/s)
3 orbits complete (3000 orbits/s)
7 orbits complete (2333 orbits/s)
7 orbits complete (2333 orbits/s)
2 orbits complete (2000 orbits/s)
8 orbits complete (2667 orbits/s)
7 orbits complete (2333 orbits/s)
9 orbits complete (4500 orbits/s)
8 orbits complete (4000 orbits/s)
9 orbits complete (3000 orbits/s)
7 orbits complete (3500 orbits/s)
10 orbits

<Figure size 800x600 with 0 Axes>